# Supervisor run — 2026-08-16 (Run 2)

Numbers and figures only. No interpretation. Every output saved to `outputs/`
with the cell-ID prefix. Summary table: `outputs/RESULTS_SUMMARY.md`.
Scope: everything except Directionality (doesn't exist yet).

Cell IDs (A1, B3, D2, ...) are dependency/build labels, not narrative order --
the notebook below is arranged as the argument (§0-§3), not by label.

## Setup

In [ ]:
import warnings, re, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"
RUN_DIR = ANALYSIS_DIR / "supervisor_run_2026-08-16"
OUT_DIR = RUN_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ERRORS = []       # (cell_id, error_str) -- logged, never raised
RESULTS = {}      # headline_number_key -> value, dumped to RESULTS_SUMMARY.md

def save_csv(cell_id, name, df):
    p = OUT_DIR / f"{cell_id}_{name}.csv"
    df.to_csv(p, index=True)
    print(f"saved {p}")

def save_fig(cell_id, name):
    p = OUT_DIR / f"{cell_id}_{name}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"saved {p}")

def wilson(successes, n):
    if n == 0:
        return np.nan, np.nan, np.nan
    rate = successes / n
    lo, hi = proportion_confint(successes, n, method="wilson")
    return rate, lo, hi

def add_wilson_ci(df, k_col, n_col):
    rates = df.apply(lambda r: pd.Series(wilson(r[k_col], r[n_col]), index=["rate", "ci_lo", "ci_hi"]), axis=1)
    return df.join(rates)

def run_step(cell_id, fn):
    try:
        fn()
    except Exception as e:
        ERRORS.append((cell_id, repr(e)))
        print(f"[SKIPPED -- {cell_id}] {e!r}")

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet")
political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")
socio_political = pd.read_csv(SOCIO_DIR / "socio_political_data.csv")
print(f"corpus: {corpus.shape}, political: {political.shape}, socio_political: {socio_political.shape}")

In [ ]:
# review_year_est: actual review_timestamp, recovered from relative-time text where missing
# (identical recovery method already used in descriptive_analysis.ipynb -- reused, not redesigned)
REL_TIME_PATTERN = re.compile(r"^(?:Edited\s+)?(a|an|\d+)\s+(year|month|week|day|hour|minute)s?\s+ago$", re.IGNORECASE)
UNIT_TO_DAYS = {"year": 365.25, "month": 30.44, "week": 7, "day": 1, "hour": 1/24, "minute": 1/(24*60)}

def parse_relative_time(text):
    m = REL_TIME_PATTERN.match(text) if isinstance(text, str) else None
    if not m:
        return None
    qty_raw, unit = m.groups()
    qty = 1 if qty_raw.lower() in ("a", "an") else int(qty_raw)
    return pd.Timedelta(days=qty * UNIT_TO_DAYS[unit.lower()])

missing_ts_mask = corpus["review_timestamp"].isna()
deltas = corpus.loc[missing_ts_mask, "review_time"].apply(parse_relative_time)
base = corpus["review_timestamp"].dt.tz_localize(None).astype("datetime64[us]")
estimated = pd.Series(pd.NaT, index=corpus.index, dtype="datetime64[us]")
raw_est = (corpus.loc[missing_ts_mask, "place_retrieved_time"] - deltas).astype("datetime64[us]")
estimated.loc[missing_ts_mask] = raw_est
corpus["review_timestamp_est"] = base.fillna(estimated)
corpus["timestamp_is_estimated"] = missing_ts_mask & corpus["review_timestamp_est"].notna()
corpus["review_year"] = corpus["review_timestamp_est"].dt.year
corpus["review_month"] = corpus["review_timestamp_est"].dt.month

SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
corpus["meteorological_season"] = corpus["review_month"].map(SEASON_MAP)
POOL_SEASON_MAP = {m: ("outdoor" if 5 <= m <= 9 else "indoor") for m in range(1, 13)}
corpus["pool_season"] = corpus["review_month"].map(POOL_SEASON_MAP)

print(f"timestamp (actual+estimated) coverage: {corpus['review_timestamp_est'].notna().mean():.2%}")

social_mask = corpus["social"] == "yes"
social_df = corpus.loc[social_mask].copy()
racialised_df = corpus.loc[corpus["racialised"] == "yes"].copy()
print(f"social=yes: {len(social_df):,}   racialised=yes: {len(racialised_df):,}")

DIMENSIONS_ALL = ["approach", "theme", "connotation", "context"]
DIMENSIONS = ["theme", "connotation", "context"]  # Approach excluded from this run per brief

In [ ]:
# constituency_code -- rebuild the place -> constituency spatial join, exactly as in
# descriptive_analysis.ipynb (broadcast place-level join back onto all review rows)
import geopandas as gpd

places = corpus[["unique_identifier", "place_name", "latitude", "longitude"]].drop_duplicates(subset="unique_identifier").reset_index(drop=True)
places_geo = gpd.GeoDataFrame(places, geometry=gpd.points_from_xy(places["longitude"], places["latitude"]), crs="EPSG:4326").to_crs("EPSG:25832")

constituencies = gpd.read_file(SHAPEFILE_PATH)
if (~constituencies.geometry.is_valid).any():
    constituencies["geometry"] = constituencies.geometry.buffer(0)

germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)
inside_mask = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_mask)].copy()

joined = gpd.sjoin_nearest(places_inside, constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]], how="left")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")
place_constituency = joined.rename(columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"})[
    ["unique_identifier", "constituency_code", "constituency", "state"]
].reset_index(drop=True)

corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")
social_df = corpus.loc[social_mask].copy()
racialised_df = corpus.loc[corpus["racialised"] == "yes"].copy()
print(f"review rows without a constituency match: {corpus['constituency_code'].isna().sum():,}")

In [ ]:
# ethnicity_final / ethnicity_group5 / ethnicity_group4 -- built exactly as in descriptive_analysis.ipynb
ARAB_LEAGUE_VARIANTS = {
    "Algeria": ["Algeria", "People's Democratic Republic of Algeria"], "Bahrain": ["Bahrain", "Kingdom of Bahrain"],
    "Comoros": ["Comoros", "Union of the Comoros"], "Djibouti": ["Djibouti", "Republic of Djibouti"],
    "Egypt": ["Egypt", "Egypt, Arab Republic of"], "Iraq": ["Iraq", "Republic of Iraq"],
    "Jordan": ["Jordan", "Hashemite Kingdom of Jordan"], "Kuwait": ["Kuwait", "State of Kuwait"],
    "Lebanon": ["Lebanon", "Lebanese Republic"], "Libya": ["Libya", "Libyan Arab Jamahiriya", "State of Libya"],
    "Mauritania": ["Mauritania", "Islamic Republic of Mauritania"], "Morocco": ["Morocco", "Kingdom of Morocco"],
    "Oman": ["Oman", "Sultanate of Oman"],
    "Palestine": ["Palestine, State of", "State of Palestine", "Palestinian Territory, Occupied"],
    "Qatar": ["Qatar", "State of Qatar"], "Saudi Arabia": ["Saudi Arabia", "Kingdom of Saudi Arabia"],
    "Somalia": ["Somalia", "Federal Republic of Somalia"], "Sudan": ["Sudan", "Republic of the Sudan"],
    "Syria": ["Syrian Arab Republic", "Syria"], "Tunisia": ["Tunisia", "Tunisian Republic"],
    "United Arab Emirates": ["United Arab Emirates"], "Yemen": ["Yemen", "Republic of Yemen"],
}

def build_ethnicity_final(country_series, n_top=10):
    present_values = set(country_series.dropna().unique())
    arab_value_map = {v: c for c, variants in ARAB_LEAGUE_VARIANTS.items() for v in variants if v in present_values}
    arab_mask = country_series.isin(arab_value_map.keys())
    non_arab_mask = ~arab_mask & country_series.notna()
    top_countries = country_series.loc[non_arab_mask].value_counts().head(n_top).index
    final = pd.Series("Other", index=country_series.index)
    final = final.mask(country_series.isin(top_countries), country_series)
    final = final.mask(arab_mask, "Arab")
    final = final.mask(country_series.isna(), "Unknown")
    return final

social_df["ethnicity_final"] = build_ethnicity_final(social_df["ethnicity_country_name"])

def to_group5(v):
    return v if v in ("Germany", "Arab", "Türkiye", "Unknown") else "Other"

EUROPE_COUNTRIES = {"Russian Federation", "Italy", "Poland", "France"}
def to_group4(v):
    if v in ("Germany", "Arab", "Türkiye", "Unknown"):
        return v
    if v in EUROPE_COUNTRIES:
        return "Europe"
    return "Other"

social_df["ethnicity_group5"] = social_df["ethnicity_final"].map(to_group5)
social_df["ethnicity_group4"] = social_df["ethnicity_final"].map(to_group4)
racialised_df = racialised_df.join(social_df[["ethnicity_final", "ethnicity_group5", "ethnicity_group4"]])
GROUP5_ORDER = ["Germany", "Türkiye", "Arab", "Other", "Unknown"]
print(social_df["ethnicity_group5"].value_counts())

In [ ]:
# const_agg -- canonical constituency-level table, built exactly as in descriptive_analysis.ipynb
backbone = political[["state", "constituency_code", "constituency"]].copy()
agg = (
    corpus.groupby("constituency_code")
    .agg(pool_count=("unique_identifier", "nunique"),
         total_text_reviews=("review_id", "count"),
         social_reviews=("social", lambda s: (s == "yes").sum()),
         racialised_reviews=("racialised", lambda s: (s == "yes").sum()))
    .reset_index()
)
const_agg = backbone.merge(agg, on="constituency_code", how="left")
for c in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[c] = const_agg[c].fillna(0).astype(int)

social_pools = corpus.loc[corpus["social"] == "yes"].groupby("constituency_code")["unique_identifier"].nunique().rename("pools_with_social")
const_agg = const_agg.merge(social_pools, on="constituency_code", how="left")
const_agg["pools_with_social"] = const_agg["pools_with_social"].fillna(0).astype(int)

MIN_SOCIAL_REVIEWS, MIN_RACIALISED_REVIEWS = 100, 10
const_agg["reliable_n"] = (const_agg["social_reviews"] >= MIN_SOCIAL_REVIEWS) & (const_agg["racialised_reviews"] >= MIN_RACIALISED_REVIEWS)

rates = const_agg.apply(lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["racialised_rate", "ci_lo", "ci_hi"]), axis=1)
const_agg = pd.concat([const_agg, rates], axis=1)
const_agg["ci_width"] = const_agg["ci_hi"] - const_agg["ci_lo"]

pool_racialised = racialised_df.groupby(["constituency_code", "unique_identifier"]).size().reset_index(name="pool_racialised_count")
pool_agg = pool_racialised.groupby("constituency_code").agg(pools_with_racialised=("unique_identifier", "nunique"),
                                                             top_pool_racialised_count=("pool_racialised_count", "max"))
const_agg = const_agg.merge(pool_agg, on="constituency_code", how="left")
const_agg["pools_with_racialised"] = const_agg["pools_with_racialised"].fillna(0).astype(int)
const_agg["pct_pools_with_racialised"] = const_agg["pools_with_racialised"] / const_agg["pools_with_social"].replace(0, np.nan)
const_agg["top_pool_share_of_racialised"] = const_agg["top_pool_racialised_count"] / const_agg["racialised_reviews"].replace(0, np.nan)

print(f"const_agg: {const_agg.shape}, reliable_n={const_agg['reliable_n'].sum()}/299")

In [ ]:
# analysis_df -- merge socio_political_data (has all derived per-1k/pct columns), plus East/split-city flags
analysis_df = const_agg.merge(socio_political, on=["state", "constituency_code", "constituency"], how="left")
analysis_df["racialised_rate_of_all"] = analysis_df["racialised_reviews"] / analysis_df["total_text_reviews"]
analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])
analysis_df["log_social_reviews"] = np.log(analysis_df["social_reviews"])
analysis_df["log_total_text_reviews"] = np.log(analysis_df["total_text_reviews"])
analysis_df["disposable_income_1k"] = analysis_df["disposable_income_eur_per_inhabitant"] / 1000

EAST_STATES = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
analysis_df["is_east"] = analysis_df["state"].isin(EAST_STATES).astype(int)

# split-city detection -- identical check_cols to redo_socio_political_relations.ipynb
CHECK_COLS = ["unemployment_total_pct", "sgb_ii_recipients_per_1k", "school_grads_higher_education_entrance_pct",
              "age_60_74_pct", "age_75_plus_pct", "disposable_income_1k"]
dup_groups = analysis_df.groupby(CHECK_COLS)["constituency"].apply(list)
dup_groups = dup_groups[dup_groups.apply(len) > 1]
flagged = {c for consts in dup_groups for c in consts}
analysis_df["in_split_city"] = analysis_df["constituency"].isin(flagged).astype(int)

for c in ["foreign_population_pct", "log_population_density", "sgb_ii_recipients_per_1k"]:
    analysis_df[c + "_c"] = analysis_df[c] - analysis_df[c].mean()

FULL_FORMULA = "racialised_reviews ~ foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east + in_split_city"

print(f"analysis_df: {analysis_df.shape}, is_east: {analysis_df['is_east'].sum()}, in_split_city: {analysis_df['in_split_city'].sum()}")

# §0 — DEFINE THE OUTCOME

### A1 / REV-A1 — Non-social reviews: is the exclusion justified?

In [ ]:
# A1a -- sample 1500 non-social reviews
nonsocial = corpus.loc[corpus["social"] == "no"].copy()
sample_a1 = nonsocial.sample(n=1500, random_state=42).copy()
sample_a1["text_for_classification"] = sample_a1["trans_review"].where(sample_a1["trans_review"].notna(), sample_a1["org_review"])
print(f"non-social pool: {len(nonsocial):,}; sampled: {len(sample_a1)}")

In [ ]:
# A1b -- run the Stage-2 racial classifier (live API call) on the 1500-review sample.
# Cached after the first successful run (same random_state=42 sample -> same items) so
# reruns of this notebook don't re-spend real API budget on identical classification.
def _a1_run():
    global sample_a1, RESULTS
    CACHE_PATH = OUT_DIR / "A1_classified_cache.csv"

    if CACHE_PATH.exists():
        cache = pd.read_csv(CACHE_PATH, dtype={"review_id": str, "racialised_a1": str})
        sample_a1 = sample_a1.merge(cache[["review_id", "racialised_a1"]], on="review_id", how="left")
        print(f"loaded cached classification for {sample_a1['racialised_a1'].notna().sum()} / {len(sample_a1)} reviews from {CACHE_PATH}")
    else:
        import sys, os, threading, requests
        from concurrent.futures import ThreadPoolExecutor, as_completed
        from dotenv import load_dotenv

        CLASSIFIER_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/data_annotation/stage_2_racial_reclassed")
        KEYS_ENV = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/data_annotation/stage_3_subs/approach/.env")
        sys.path.insert(0, str(CLASSIFIER_DIR))
        load_dotenv(KEYS_ENV, override=True)
        # the classifier module calls sys.stdout.reconfigure() at import time, which Jupyter's
        # ipykernel OutStream doesn't implement -- shim it before importing
        if not hasattr(sys.stdout, "reconfigure"):
            sys.stdout.reconfigure = lambda *a, **k: None
        if not hasattr(sys.stderr, "reconfigure"):
            sys.stderr.reconfigure = lambda *a, **k: None
        import racial_classifier_parallelization_examples as rc

        keys_raw = os.getenv("OPENROUTER_API_KEYS") or os.getenv("OPENROUTER_API_KEY")
        all_keys = [k.strip() for k in keys_raw.split(",") if k.strip()]

        # cheap pre-flight: OpenRouter's /key endpoint reports balance with no completion cost --
        # filter out dead/exhausted keys before spending anything on classification
        api_keys = []
        for k in all_keys:
            try:
                info = requests.get("https://openrouter.ai/api/v1/key", headers={"Authorization": f"Bearer {k}"}, timeout=15)
                if info.status_code == 200:
                    d = info.json().get("data", {})
                    limit, usage = d.get("limit"), d.get("usage", 0)
                    if limit is None or usage < limit:
                        api_keys.append(k)
            except Exception:
                pass
        print(f"usable keys: {len(api_keys)} / {len(all_keys)}")
        n_workers = max(1, len(api_keys))
        tracker = rc.CostTracker(budget_limit=50.0)
        key_tracker = rc.ApiKeyTracker(api_keys)

        items = list(zip(sample_a1["review_id"], sample_a1["text_for_classification"]))
        results = {}

        def classify_one(idx_item):
            idx, (rid, text) = idx_item
            api_key = api_keys[idx % n_workers]
            if not isinstance(text, str) or not text.strip():
                return rid, None
            r = rc.classify_single(rid, text, api_key, tracker, key_tracker)
            return rid, (r.get("racialised") if r else None)

        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futures = [ex.submit(classify_one, (i, item)) for i, item in enumerate(items)]
            for f in as_completed(futures):
                rid, label = f.result()
                results[rid] = label

        sample_a1["racialised_a1"] = sample_a1["review_id"].map(results)
        print(tracker.summary())
        pd.DataFrame({"review_id": sample_a1["review_id"], "racialised_a1": sample_a1["racialised_a1"]}).to_csv(CACHE_PATH, index=False)
        print(f"cached classification to {CACHE_PATH}")

    n_unclassified = sample_a1["racialised_a1"].isna().sum()
    print(f"unclassified (API failure or empty text): {n_unclassified}")

    n_hits = (sample_a1["racialised_a1"] == "yes").sum()
    n_valid = sample_a1["racialised_a1"].notna().sum()
    rate = n_hits / n_valid if n_valid else float("nan")
    print(f"\nnon-social sample n:        {len(sample_a1)}")
    print(f"racialised hits:            {n_hits}")
    print(f"rate:                       {rate:.2%}")
    print(f"comparison: social rate     1.54%")
    RESULTS["A1_nonsocial_n"] = len(sample_a1)
    RESULTS["A1_nonsocial_hits"] = int(n_hits)
    RESULTS["A1_nonsocial_rate_pct"] = round(rate * 100, 3)

run_step("A1b", _a1_run)

In [ ]:
# A1c -- export every hit + 30 random non-hits (spot-check on Stage 1), per spec columns
def _a1_export():
    hits = sample_a1.loc[sample_a1["racialised_a1"] == "yes"].copy()
    non_hits_pool = sample_a1.loc[sample_a1["racialised_a1"] == "no"]
    spot_check = non_hits_pool.sample(n=min(30, len(non_hits_pool)), random_state=42).copy()

    hits["flag"] = "racialised_hit"
    spot_check["flag"] = "non_hit_spotcheck"
    export = pd.concat([hits, spot_check])[["review_id", "org_review", "trans_review", "place_name", "review_year", "flag"]]
    export = export.rename(columns={"org_review": "text", "trans_review": "translated_text", "place_name": "pool_name", "review_year": "year"})
    export.to_csv(OUT_DIR / "A1_nonsocial_hits.csv", index=False)
    print(f"exported {len(export)} rows ({len(hits)} hits + {len(spot_check)} spot-check) to A1_nonsocial_hits.csv")

run_step("A1c", _a1_export)

In [ ]:
# REV-A1 -- restate the non-social result correctly. The A1 sample only detects reviews that
# are BOTH missed-by-Stage-1 AND racialised -- it cannot estimate Stage 1 accuracy overall, so no
# "Stage 1 false-negative rate" is computed here. The two counts below come from manually reading
# every hit in outputs/A1_nonsocial_hits.csv (not auto-derived from the classifier's own label).
def _rev_a1():
    genuine_nonsocial_racialisation = 0          # hits confirmed genuinely off-topic (food/facilities etc.) on manual read
    stage1_missed_and_racialised = 1             # hits confirmed to actually be a social encounter Stage 1 missed
    n = RESULTS.get("A1_nonsocial_n", 1500)
    rate = stage1_missed_and_racialised / n

    print(f"genuine non-social racialisation, n of {n}:     {genuine_nonsocial_racialisation}")
    print(f"social reviews missed by Stage 1 and racialised: {stage1_missed_and_racialised}  ({rate:.2%})")

    table = pd.DataFrame([
        {"metric": "genuine non-social racialisation", "n": genuine_nonsocial_racialisation, "of_n": n},
        {"metric": "social reviews missed by Stage 1 and racialised", "n": stage1_missed_and_racialised, "of_n": n, "rate_pct": round(rate * 100, 2)},
    ])
    save_csv("REV-A1", "corrected_nonsocial_result", table)

    RESULTS["REV_A1_genuine_nonsocial"] = genuine_nonsocial_racialisation
    RESULTS["REV_A1_stage1_missed_racialised"] = stage1_missed_and_racialised
    RESULTS["REV_A1_stage1_missed_racialised_pct"] = round(rate * 100, 2)

run_step("REV-A1", _rev_a1)

### A2 / REV-A2 — Social share over time

In [ ]:
def _a2():
    yearly = corpus[corpus["review_year"].between(2016, 2025)].groupby("review_year").agg(
        total_reviews=("review_id", "size"),
        social_reviews=("social", lambda s: (s == "yes").sum()),
    )
    yearly["social_share"] = yearly["social_reviews"] / yearly["total_reviews"]
    save_csv("A2", "social_share_by_year", yearly)
    print(yearly)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    ax1.plot(yearly.index, yearly["social_share"], marker="o", color="#C44E52")
    ax1.set_ylabel("social share")
    ax1.set_title("Social share of reviews by year")
    ax2.bar(yearly.index, yearly["total_reviews"], color="#4C72B0")
    ax2.set_ylabel("total reviews")
    ax2.set_xlabel("review year")
    plt.tight_layout()
    save_fig("A2", "social_share_and_volume_by_year")
    plt.show()

    RESULTS["A2_social_share_2018"] = round(yearly.loc[2018, "social_share"] * 100, 2) if 2018 in yearly.index else None
    RESULTS["A2_social_share_2025"] = round(yearly.loc[2025, "social_share"] * 100, 2) if 2025 in yearly.index else None

run_step("A2", _a2)

In [ ]:
# REV-A2 -- the multiplier depends heavily on the baseline year; report the full sensitivity table
def _rev_a2():
    yearly = corpus[social_mask & corpus["review_year"].between(2016, 2025)].groupby("review_year").agg(
        social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
    )
    yearly["rate"] = yearly["racialised_reviews"] / yearly["social_reviews"]
    rate_2025 = yearly.loc[2025, "rate"]

    baselines = [2016, 2017, 2018, 2019, 2021]
    rows = []
    for b in baselines:
        r = yearly.loc[b, "rate"]
        label = f"{b} (trough)" if b == 2021 else str(b)
        rows.append({"baseline": label, "rate": r, "ratio_to_2025": rate_2025 / r})
    table = pd.DataFrame(rows)
    save_csv("REV-A2", "baseline_sensitivity", table)
    print(table.round(4))

    pre2020 = yearly.loc[2016:2019, "rate"]
    ratios_pre2020 = rate_2025 / pre2020
    print(f"\nminimum ratio across pre-2020 baselines: {ratios_pre2020.min():.3f}")
    print(f"maximum ratio across pre-2020 baselines: {ratios_pre2020.max():.3f}")

    RESULTS["REV_A2_ratio_min_pre2020"] = round(ratios_pre2020.min(), 3)
    RESULTS["REV_A2_ratio_max_pre2020"] = round(ratios_pre2020.max(), 3)
    for b in baselines:
        RESULTS[f"REV_A2_ratio_{b}"] = round(rate_2025 / yearly.loc[b, "rate"], 3)

run_step("REV-A2", _rev_a2)

### A3 — Social share vs. predictors

In [ ]:
def _a3():
    social_share_const = corpus.groupby("constituency_code").apply(lambda g: (g["social"] == "yes").mean(), include_groups=False).rename("social_share")
    a3_df = analysis_df.set_index("constituency_code").join(social_share_const)

    predictors = ["foreign_population_pct", "log_population_density", "sgb_ii_recipients_per_1k", "is_east"]
    rows = []
    for p in predictors:
        rho, pval = stats.spearmanr(a3_df[p], a3_df["social_share"])
        rows.append({"predictor": p, "rho": rho, "p": pval})
    corr_table = pd.DataFrame(rows)
    save_csv("A3", "social_share_correlations", corr_table)
    print(corr_table.round(4))

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(a3_df["foreign_population_pct"], a3_df["social_share"], alpha=0.6)
    ax.set_xlabel("foreign_population_pct")
    ax.set_ylabel("social share")
    ax.set_title("Social share vs. foreign population %")
    plt.tight_layout()
    save_fig("A3", "social_share_vs_foreign_population")
    plt.show()

    for _, r in corr_table.iterrows():
        RESULTS[f"A3_rho_{r['predictor']}"] = round(r["rho"], 3)

run_step("A3", _a3)

**Definition used from here on:** racialised rate = racialised reviews / social reviews
(social=yes as the denominator). A1/REV-A1 found ~0 genuine non-social racialisation lost by
excluding social=no; A2/REV-A2 and A3 describe the denominator's own behaviour over time and
across constituencies without adjusting it.

# §1 — RQ1: PREVALENCE, FORM, CHANGE

### B1 — Prevalence

In [ ]:
def _b1():
    overall_rate, lo, hi = wilson(racialised_df.shape[0], social_df.shape[0])
    rate_all = racialised_df.shape[0] / corpus.shape[0]
    n_pools_racialised = racialised_df["unique_identifier"].nunique()

    summary = pd.DataFrame([
        {"metric": "rate (social denominator)", "value": overall_rate, "ci_lo": lo, "ci_hi": hi},
        {"metric": "rate (all-reviews denominator)", "value": rate_all, "ci_lo": np.nan, "ci_hi": np.nan},
        {"metric": "pools with >=1 racialised review", "value": n_pools_racialised, "ci_lo": np.nan, "ci_hi": np.nan},
    ])
    save_csv("B1", "prevalence_summary", summary)
    print(summary)

    print("\nconstituency racialised_rate distribution:")
    print(const_agg["racialised_rate"].describe())
    save_csv("B1", "constituency_rate_distribution", const_agg["racialised_rate"].describe().to_frame())

    print("\nsocial reviews per constituency -- min/median/max:")
    print(const_agg["social_reviews"].agg(["min", "median", "max"]))

    RESULTS["B1_rate_social_denom_pct"] = round(overall_rate * 100, 3)
    RESULTS["B1_rate_social_denom_ci"] = f"[{lo:.4f}, {hi:.4f}]"
    RESULTS["B1_rate_all_denom_pct"] = round(rate_all * 100, 3)
    RESULTS["B1_racialised_reviews"] = int(racialised_df.shape[0])
    RESULTS["B1_pools_with_racialised"] = int(n_pools_racialised)
    RESULTS["B1_const_rate_min_med_max"] = tuple(const_agg["social_reviews"].agg(["min", "median", "max"]))

run_step("B1", _b1)

### D1 — Properties of the outcome

In [ ]:
def _d1():
    count = analysis_df["racialised_reviews"]
    rate = analysis_df["racialised_rate"]
    social = analysis_df["social_reviews"]

    count_stats = pd.Series({
        "n": count.shape[0], "mean": count.mean(), "variance": count.var(), "variance/mean": count.var() / count.mean(),
        "min": count.min(), "median": count.median(), "max": count.max(), "skew": count.skew(), "n_zeros": int((count == 0).sum()),
    })
    rate_stats = pd.Series({
        "n": rate.shape[0], "mean": rate.mean(), "variance": rate.var(), "sd": rate.std(),
        "min": rate.min(), "q25": rate.quantile(.25), "median": rate.median(), "q75": rate.quantile(.75),
        "max": rate.max(), "skew": rate.skew(),
    })
    social_stats = pd.Series({
        "n": social.shape[0], "mean": social.mean(), "min": social.min(), "q25": social.quantile(.25),
        "median": social.median(), "q75": social.quantile(.75), "max": social.max(),
    })

    save_csv("D1", "count_properties", count_stats.to_frame("value"))
    save_csv("D1", "rate_properties", rate_stats.to_frame("value"))
    save_csv("D1", "social_reviews_properties", social_stats.to_frame("value"))
    print("--- racialised_reviews (count) ---"); print(count_stats.round(4))
    print("\n--- racialised_rate ---"); print(rate_stats.round(5))
    print("\n--- social_reviews ---"); print(social_stats.round(2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(count, bins=30, color="steelblue", edgecolor="black")
    axes[0].set_title("racialised_reviews (count)"); axes[0].set_xlabel("count")
    axes[1].hist(rate, bins=30, color="salmon", edgecolor="black")
    axes[1].set_title("racialised_rate"); axes[1].set_xlabel("rate")
    plt.tight_layout()
    save_fig("D1", "count_and_rate_histograms")
    plt.show()

    RESULTS["D1_variance_over_mean"] = round(count_stats["variance/mean"], 3)

run_step("D1", _d1)

### A4 / D5 — Is constituency variation greater than sampling noise? (dispersion ladder)

In [ ]:
def _a4_d5():
    df = analysis_df.copy()
    offset = np.log(df["social_reviews"])

    null_const = smf.glm("racialised_reviews ~ 1", data=df, family=sm.families.Poisson(), offset=offset).fit()
    disp_null_const = null_const.pearson_chi2 / null_const.df_resid

    full = smf.glm(FULL_FORMULA, data=df, family=sm.families.Poisson(), offset=offset).fit()
    disp_full = full.pearson_chi2 / full.df_resid

    yearly = corpus[(corpus["social"] == "yes") & corpus["review_year"].between(2016, 2025)].groupby("review_year").agg(
        racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
        social_reviews=("racialised", "size"),
    )
    yearly = yearly[yearly["social_reviews"] >= 20]
    null_year = smf.glm("racialised_reviews ~ 1", data=yearly, family=sm.families.Poisson(), offset=np.log(yearly["social_reviews"])).fit()
    disp_null_year = null_year.pearson_chi2 / null_year.df_resid

    prop_explained = 1 - (disp_full - 1) / (disp_null_const - 1)

    print(f"dispersion, intercept-only (constituency):   {disp_null_const:.3f}")
    print(f"dispersion, with predictors:                 {disp_full:.3f}")
    print(f"proportion of excess variation explained:    {prop_explained:.3f}")
    print(f"dispersion, intercept-only (year):           {disp_null_year:.3f}")

    RESULTS["A4_dispersion_null_constituency"] = round(disp_null_const, 3)
    RESULTS["A4_dispersion_full"] = round(disp_full, 3)
    RESULTS["A4_prop_excess_explained"] = round(prop_explained, 3)
    RESULTS["A4_dispersion_null_year"] = round(disp_null_year, 3)

    # D5 -- restate as one ladder table, four figures, clearly labelled as separate tests
    ladder = pd.DataFrame([
        {"model": "intercept only", "grouping": "constituency", "dispersion": disp_null_const, "df": null_const.df_resid},
        {"model": "full model", "grouping": "constituency", "dispersion": disp_full, "df": full.df_resid},
        {"model": "intercept only", "grouping": "year", "dispersion": disp_null_year, "df": null_year.df_resid},
        {"model": "full model", "grouping": "year", "dispersion": np.nan, "df": np.nan},
    ])
    save_csv("D5", "dispersion_ladder", ladder)
    print("\n--- D5 dispersion ladder ---")
    print(ladder)

    total_social = df["social_reviews"].sum()
    mean_per_year = total_social / 10
    mean_per_const = total_social / 299
    print(f"\ntotal social reviews:            {total_social}")
    print(f"mean per year (÷10):              {mean_per_year:.1f}")
    print(f"mean per constituency (÷299):     {mean_per_const:.1f}")
    print(f"ratio:                            {mean_per_year / mean_per_const:.3f}")
    RESULTS["D5_mean_per_year"] = round(mean_per_year, 1)
    RESULTS["D5_mean_per_constituency"] = round(mean_per_const, 1)
    RESULTS["D5_ratio_year_to_const"] = round(mean_per_year / mean_per_const, 3)

run_step("A4-D5", _a4_d5)

### B2 — Form (Theme, Context, Connotation; Approach excluded)

In [ ]:
def _b2():
    for dim in DIMENSIONS:
        exploded = racialised_df[dim].explode()
        counts = exploded.value_counts()
        pct = counts / len(racialised_df)
        table = pd.DataFrame({"n_reviews": counts, "pct_of_racialised_reviews": (pct * 100).round(1)})
        save_csv("B2", f"{dim}_distribution", table)
        n_multi = dim == "theme"
        label = "multi-label (<=3)" if n_multi else "single-label"
        print(f"--- {dim} [{label}] ---")
        print(table)

run_step("B2", _b2)

### B3 — Rate by year

In [ ]:
def _b3():
    yearly = corpus[social_mask & corpus["review_year"].between(2016, 2025)].groupby("review_year").agg(
        social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
    )
    yearly_all = corpus[corpus["review_year"].between(2016, 2025)].groupby("review_year").agg(total_reviews=("review_id", "size"))
    yearly = yearly.join(yearly_all)
    yearly["rate_all_denom"] = yearly["racialised_reviews"] / yearly["total_reviews"]
    yearly = add_wilson_ci(yearly, "racialised_reviews", "social_reviews")
    save_csv("B3", "rate_by_year", yearly)
    print(yearly.round(5))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(yearly.index, yearly["rate"], marker="o", color="#C44E52", label="rate (social denom)")
    ax.fill_between(yearly.index, yearly["ci_lo"], yearly["ci_hi"], alpha=0.2, color="#C44E52")
    ax.plot(yearly.index, yearly["rate_all_denom"], marker="s", color="#4C72B0", label="rate (all-reviews denom)")
    ax.set_ylabel("racialised rate")
    ax.set_xlabel("review year")
    ax.legend()
    plt.tight_layout()
    save_fig("B3", "rate_by_year")
    plt.show()

    for y in (2018, 2021, 2025):
        if y in yearly.index:
            RESULTS[f"B3_rate_{y}_pct"] = round(yearly.loc[y, "rate"] * 100, 3)

run_step("B3", _b3)

### A5 — Ethnicity composition of social reviews by year

In [ ]:
def _a5():
    d = social_df[social_df["review_year"].between(2016, 2025)].copy()

    for grouping, order in [("ethnicity_group5", GROUP5_ORDER), ("ethnicity_final", None)]:
        counts = d.groupby(["review_year", grouping]).size().unstack(fill_value=0)
        if order:
            counts = counts.reindex(columns=order)
        shares = counts.div(counts.sum(axis=1), axis=0)

        save_csv("A5", f"{grouping}_share_by_year", shares)
        save_csv("A5", f"{grouping}_counts_by_year", counts)
        print(f"--- {grouping} shares ---")
        print(shares.round(3))
        print(f"--- {grouping} raw counts ---")
        print(counts)

        fig, ax = plt.subplots(figsize=(10, 6))
        shares.plot.area(ax=ax, cmap="tab10")
        ax.set_ylabel("share of social reviews")
        ax.set_title(f"{grouping} composition of social reviews by year")
        plt.tight_layout()
        save_fig("A5", f"{grouping}_share_by_year")
        plt.show()

run_step("A5", _a5)

### A5x — Composition vs. behaviour decomposition

In [ ]:
# A5x-1 -- racialised rate by group by year
def _a5x1():
    global rate_gy, social_gy
    soc = social_df[social_df["review_year"].between(2016, 2025)]
    tab = soc.groupby(["review_year", "ethnicity_group5"]).agg(
        social=("review_id", "size"), rac=("racialised", lambda s: (s == "yes").sum())
    )
    tab["rate"] = tab["rac"] / tab["social"]
    rate_gy = tab["rate"].unstack().reindex(columns=GROUP5_ORDER)
    social_gy = tab["social"].unstack().reindex(columns=GROUP5_ORDER)
    rac_gy = tab["rac"].unstack().reindex(columns=GROUP5_ORDER)

    save_csv("A5x-1", "rate_by_group_year", rate_gy)
    save_csv("A5x-1", "social_count_by_group_year", social_gy)
    save_csv("A5x-1", "rac_count_by_group_year", rac_gy)
    print("--- rate by group by year ---"); print(rate_gy.round(4))
    print("\n--- social n by group by year ---"); print(social_gy)

    unreliable = (social_gy < 500).sum().sum()
    print(f"\ncells with social < 500 (unreliable): {unreliable}")

run_step("A5x-1", _a5x1)

In [ ]:
# A5x-2 -- group-level overall rates (2016-2025)
def _a5x2():
    global overall_by_group
    soc = social_df[social_df["review_year"].between(2016, 2025)]
    overall_by_group = soc.groupby("ethnicity_group5").apply(
        lambda g: (g["racialised"] == "yes").sum() / len(g), include_groups=False
    ).reindex(GROUP5_ORDER)
    save_csv("A5x-2", "overall_rate_by_group", overall_by_group.to_frame("rate"))
    print(overall_by_group.round(4))

run_step("A5x-2", _a5x2)

In [ ]:
# A5x-3 -- composition-only counterfactual: each group's rate held fixed at its overall value
def _a5x3():
    global comp, counterfactual
    soc = social_df[social_df["review_year"].between(2016, 2025)]
    counts = soc.groupby(["review_year", "ethnicity_group5"]).size().unstack(fill_value=0).reindex(columns=GROUP5_ORDER)
    comp = counts.div(counts.sum(axis=1), axis=0)
    counterfactual = (comp * overall_by_group).sum(axis=1)
    save_csv("A5x-3", "composition_only_counterfactual", counterfactual.to_frame("counterfactual_rate"))
    print(counterfactual.round(5))

run_step("A5x-3", _a5x3)

In [ ]:
# A5x-4 -- standardised series: composition frozen at a base year, each group's rate varies
def _a5x4():
    global standardised_by_base
    standardised_by_base = {}
    for base_year in [2018, 2019, 2022]:
        standardised_by_base[base_year] = (rate_gy * comp.loc[base_year]).sum(axis=1)
    table = pd.DataFrame(standardised_by_base)
    table.columns = [f"standardised_base_{y}" for y in table.columns]
    save_csv("A5x-4", "standardised_series", table)
    print(table.round(5))

run_step("A5x-4", _a5x4)

In [ ]:
# A5x-5 -- log decomposition on matched years
def _a5x5():
    observed = social_df[social_df["review_year"].between(2016, 2025)].groupby("review_year").apply(
        lambda g: (g["racialised"] == "yes").sum() / len(g), include_groups=False
    )

    def decompose(base_year, end_year=2025):
        tot = np.log(observed[end_year] / observed[base_year])
        comp_eff = np.log(counterfactual[end_year] / counterfactual[base_year])
        return tot, comp_eff

    rows = []
    for base in (2018, 2021):
        tot, comp_eff = decompose(base)
        rows.append({
            "baseline": base,
            "observed_ratio": np.exp(tot),
            "counterfactual_ratio": np.exp(comp_eff),
            "implied_behaviour_ratio": np.exp(tot - comp_eff),
            "composition_share_of_log_rise": comp_eff / tot,
            "behaviour_share_of_log_rise": 1 - comp_eff / tot,
        })
    table = pd.DataFrame(rows)
    save_csv("A5x-5", "log_decomposition", table)
    print(table.round(4))

    for _, r in table.iterrows():
        RESULTS[f"A5x5_comp_share_base{int(r['baseline'])}"] = round(r["composition_share_of_log_rise"], 3)
        RESULTS[f"A5x5_behav_share_base{int(r['baseline'])}"] = round(r["behaviour_share_of_log_rise"], 3)

run_step("A5x-5", _a5x5)

In [ ]:
# A5x-6 -- figures: three-line decomposition panel, and group share indexed to 2016
def _a5x6():
    observed = social_df[social_df["review_year"].between(2016, 2025)].groupby("review_year").apply(
        lambda g: (g["racialised"] == "yes").sum() / len(g), include_groups=False
    )
    standardised_2018 = standardised_by_base[2018]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(observed.index, observed.values, marker="o", label="observed")
    ax.plot(standardised_2018.index, standardised_2018.values, marker="s", label="standardised (2018 composition)")
    ax.plot(counterfactual.index, counterfactual.values, marker="^", label="counterfactual (composition-only)")
    ax.set_xlabel("review year"); ax.set_ylabel("racialised rate"); ax.legend()
    plt.tight_layout()
    save_fig("A5x-6", "observed_standardised_counterfactual")
    plt.show()

    indexed = comp.div(comp.loc[2016])
    fig, ax = plt.subplots(figsize=(9, 5))
    for g in GROUP5_ORDER:
        ax.plot(indexed.index, indexed[g], marker="o", label=g)
    ax.axhline(1.0, color="gray", ls="--", lw=1)
    ax.set_xlabel("review year"); ax.set_ylabel("share indexed to 2016 = 1.0"); ax.legend()
    plt.tight_layout()
    save_fig("A5x-6", "group_share_indexed_2016")
    plt.show()

run_step("A5x-6", _a5x6)

### Y1 / Y2 — Model the yearly trend

In [ ]:
# Y1 -- negative binomial on year (10 observations), three specifications
def _y1():
    yr = social_df[social_df["review_year"].between(2016, 2025)].groupby("review_year").agg(
        social=("review_id", "size"), rac=("racialised", lambda s: (s == "yes").sum())
    ).reset_index()
    yr["t"] = yr["review_year"] - 2016
    yr["covid"] = yr["review_year"].isin([2020, 2021]).astype(int)
    yr["post2021"] = (yr["review_year"] >= 2022).astype(int)
    offset = np.log(yr["social"])

    def fit_nb(formula):
        p = smf.glm(formula, data=yr, family=sm.families.Poisson(), offset=offset).fit()
        mu = p.mu
        aux = sm.OLS(((yr["rac"] - mu) ** 2 - mu) / mu, mu).fit()
        alpha = max(float(aux.params.iloc[0]), 1e-6)
        return smf.glm(formula, data=yr, family=sm.families.NegativeBinomial(alpha=alpha), offset=offset).fit()

    specs = {"Y1a: rac~t": "rac ~ t", "Y1b: rac~t+covid": "rac ~ t + covid", "Y1c: rac~t+post2021": "rac ~ t + post2021"}
    rows = []
    for label, f in specs.items():
        m = fit_nb(f)
        ci = m.conf_int().loc["t"]
        rows.append({"spec": label, "rate_ratio_t": np.exp(m.params["t"]), "ci_lo": np.exp(ci[0]),
                     "ci_hi": np.exp(ci[1]), "p": m.pvalues["t"], "AIC": m.aic})
    table = pd.DataFrame(rows)
    save_csv("Y1", "year_level_trend_models", table)
    print(table.round(4))
    print("\nnote: n=10 year-observations -- descriptive fits, wide CIs.")
    RESULTS["Y1a_rate_ratio_t"] = round(table.iloc[0]["rate_ratio_t"], 4)

run_step("Y1", _y1)

In [ ]:
# Y2 -- negative binomial on the constituency-year panel, clustered SEs by constituency
def _y2():
    panel = social_df[social_df["review_year"].between(2016, 2025)].groupby(["constituency_code", "review_year"]).agg(
        social=("review_id", "size"), rac=("racialised", lambda s: (s == "yes").sum())
    ).reset_index()
    panel = panel[panel["social"] >= 30].copy()
    panel["t"] = panel["review_year"] - 2016
    offset = np.log(panel["social"])

    p = smf.glm("rac ~ t", data=panel, family=sm.families.Poisson(), offset=offset).fit()
    mu = p.mu
    aux = sm.OLS(((panel["rac"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha = max(float(aux.params.iloc[0]), 1e-6)
    m = smf.glm("rac ~ t", data=panel, family=sm.families.NegativeBinomial(alpha=alpha), offset=offset).fit(
        cov_type="cluster", cov_kwds={"groups": panel["constituency_code"]})

    rr = np.exp(m.params["t"])
    ci = np.exp(m.conf_int().loc["t"])
    print(f"rate ratio on t: {rr:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]  p={m.pvalues['t']:.4e}")
    print(f"n rows: {len(panel)}, n constituencies: {panel['constituency_code'].nunique()}")

    table = pd.DataFrame([{"rate_ratio_t": rr, "ci_lo": ci[0], "ci_hi": ci[1], "p": m.pvalues["t"],
                            "n_rows": len(panel), "n_constituencies": panel["constituency_code"].nunique()}])
    save_csv("Y2", "constituency_year_panel_trend", table)
    RESULTS["Y2_rate_ratio_t"] = round(rr, 4)
    RESULTS["Y2_ci"] = f"[{ci[0]:.4f}, {ci[1]:.4f}]"

run_step("Y2", _y2)

### B4 — Content composition by year (Theme, Context; Approach excluded; denom = racialised reviews)

In [ ]:
def _b4():
    d = racialised_df[racialised_df["review_year"].between(2016, 2025)]
    for dim in ["theme", "context"]:
        exploded = d[["review_year", dim]].explode(dim)
        counts = exploded.groupby(["review_year", dim]).size().unstack(fill_value=0)
        n_per_year = d.groupby("review_year").size()
        shares = counts.div(n_per_year, axis=0)
        save_csv("B4", f"{dim}_share_by_year", shares)
        print(f"--- {dim} share of racialised reviews by year ---")
        print(shares.round(3))

        fig, ax = plt.subplots(figsize=(9, 5))
        shares.plot(ax=ax, marker="o")
        ax.set_ylabel(f"share of racialised reviews ({dim})")
        ax.set_xlabel("review year")
        plt.tight_layout()
        save_fig("B4", f"{dim}_share_by_year")
        plt.show()

        if dim == "context":
            for label in ["generalised", "incident"]:
                if label in shares.columns:
                    if 2018 in shares.index:
                        RESULTS[f"B4_{label}_2018_pct"] = round(shares.loc[2018, label] * 100, 2)
                    if 2025 in shares.index:
                        RESULTS[f"B4_{label}_2025_pct"] = round(shares.loc[2025, label] * 100, 2)

run_step("B4", _b4)

### B5 — Seasonality

In [ ]:
def _b5():
    # start year 2016 throughout; the one exception is the two year-crossing season splits
    # below, where Jan-Apr (indoor/outdoor) / Jan-May (summer/winter) of a partial 2026 rolls
    # into that season's 2025 bucket rather than being dropped or shown as its own partial year.
    ts_social_actual = corpus.loc[social_mask & ~corpus["timestamp_is_estimated"]].copy()

    by_month = ts_social_actual[ts_social_actual["review_year"].between(2016, 2025)].groupby("review_month").agg(
        social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
    )
    by_month = add_wilson_ci(by_month, "racialised_reviews", "social_reviews")
    save_csv("B5", "rate_by_month", by_month)
    print("--- rate by calendar month (years pooled, 2016-2025) ---")
    print(by_month.round(5))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(by_month.index, by_month["rate"], yerr=[by_month["rate"] - by_month["ci_lo"], by_month["ci_hi"] - by_month["rate"]], fmt="o-")
    ax.set_xlabel("calendar month")
    ax.set_ylabel("racialised rate")
    plt.tight_layout()
    save_fig("B5", "rate_by_month")
    plt.show()

    # indoor (Oct-Apr) vs outdoor (May-Sep) by year -- Jan-Apr rolls into the PREVIOUS year's
    # indoor season, since that's a continuation of the same Oct-started indoor period
    def get_pool_season_year(row):
        month, year = row["review_month"], row["review_year"]
        if 5 <= month <= 9:
            return year, "outdoor"
        elif month in (10, 11, 12):
            return year, "indoor"
        else:
            return year - 1, "indoor"

    ts_social_actual[["pool_year", "pool_season"]] = ts_social_actual.apply(lambda r: pd.Series(get_pool_season_year(r)), axis=1)
    ts_seasonal = ts_social_actual[ts_social_actual["pool_year"].between(2016, 2025)]

    by_season_year = ts_seasonal.groupby(["pool_year", "pool_season"]).agg(
        social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
    )
    by_season_year = add_wilson_ci(by_season_year, "racialised_reviews", "social_reviews")
    save_csv("B5", "indoor_outdoor_by_year", by_season_year)
    print("--- indoor vs outdoor by year ---")
    print(by_season_year.round(5))

    fig, ax = plt.subplots(figsize=(9, 5))
    for season, color in [("outdoor", "#DD8452"), ("indoor", "#4C72B0")]:
        sub = by_season_year.xs(season, level="pool_season")
        ax.errorbar(sub.index, sub["rate"], yerr=[sub["rate"] - sub["ci_lo"], sub["ci_hi"] - sub["rate"]], fmt="o-", label=season, color=color)
    ax.set_xlabel("year"); ax.set_ylabel("racialised rate"); ax.legend()
    plt.tight_layout()
    save_fig("B5", "indoor_outdoor_by_year")
    plt.show()

    # summer (Jun-Aug) vs winter (all other months) by year -- Jan-May rolls into the
    # PREVIOUS year's winter, same year-crossing logic as indoor/outdoor above
    def get_summer_winter_year(row):
        month, year = row["review_month"], row["review_year"]
        if 6 <= month <= 8:
            return year, "Summer"
        elif month in (9, 10, 11, 12):
            return year, "Winter"
        else:
            return year - 1, "Winter"

    ts_social_actual[["sw_year", "summer_vs_winter"]] = ts_social_actual.apply(lambda r: pd.Series(get_summer_winter_year(r)), axis=1)
    ts_sw = ts_social_actual[ts_social_actual["sw_year"].between(2016, 2025)]

    by_summer_year = ts_sw.groupby(["sw_year", "summer_vs_winter"]).agg(
        social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
    )
    by_summer_year = add_wilson_ci(by_summer_year, "racialised_reviews", "social_reviews")
    save_csv("B5", "summer_vs_winter_by_year", by_summer_year)
    print("--- summer vs winter by year ---")
    print(by_summer_year.round(5))

    fig, ax = plt.subplots(figsize=(9, 5))
    for season, color in [("Summer", "#DD8452"), ("Winter", "#4C72B0")]:
        sub = by_summer_year.xs(season, level="summer_vs_winter")
        ax.errorbar(sub.index, sub["rate"], yerr=[sub["rate"] - sub["ci_lo"], sub["ci_hi"] - sub["rate"]], fmt="o-", label=season, color=color)
    ax.set_xlabel("year"); ax.set_ylabel("racialised rate"); ax.legend()
    plt.tight_layout()
    save_fig("B5", "summer_vs_winter_by_year")
    plt.show()

run_step("B5", _b5)

# §2 — RQ2: WHO

### B6 — Group rates

In [ ]:
def _b6():
    grp = social_df.groupby("ethnicity_group5").agg(
        n_social=("racialised", "size"), n_racialised=("racialised", lambda s: (s == "yes").sum())
    )
    grp = add_wilson_ci(grp, "n_racialised", "n_social").sort_values("rate", ascending=False)
    save_csv("B6", "group_rates_ethnicity_group5", grp)
    print(grp.round(5))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(range(len(grp)), grp["rate"], yerr=[grp["rate"] - grp["ci_lo"], grp["ci_hi"] - grp["rate"]], fmt="o")
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp.index, rotation=30)
    ax.set_ylabel("racialised rate within group's own social reviews")
    plt.tight_layout()
    save_fig("B6", "group_rates_ethnicity_group5")
    plt.show()

run_step("B6", _b6)

**[5th dimension] Directionality — blocked.** Does not exist yet; excluded from this run per brief.

# §3 — RQ3: WHERE

### B7 — Spatial

In [ ]:
def _b7():
    global lisa, moran_gdf, w
    from libpysal.weights import Queen
    from esda.moran import Moran, Moran_Local

    map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(const_agg, on="constituency_code", how="left")
    moran_gdf = map_df.dropna(subset=["racialised_rate"]).reset_index(drop=True)
    w = Queen.from_dataframe(moran_gdf, use_index=False)
    w.transform = "r"
    moran = Moran(moran_gdf["racialised_rate"].values, w, permutations=999)
    print(f"Global Moran's I = {moran.I:.4f}, z = {moran.z_sim:.3f}, p = {moran.p_sim:.4f}")
    RESULTS["B7_moran_I"] = round(moran.I, 4)
    RESULTS["B7_moran_p"] = moran.p_sim

    lisa = Moran_Local(moran_gdf["racialised_rate"].values, w, permutations=999, seed=42)
    sig = lisa.p_sim < 0.05
    quadrant_names = {1: "High-High (hotspot)", 2: "Low-High", 3: "Low-Low (coldspot)", 4: "High-Low"}
    moran_gdf["lisa_quadrant"] = np.where(sig, pd.Series(lisa.q).map(quadrant_names), "Not significant")

    counts = moran_gdf["lisa_quadrant"].value_counts()
    save_csv("B7", "lisa_counts", counts.to_frame("n_constituencies"))
    print(counts)

    fig, ax = plt.subplots(figsize=(8, 10))
    colors = {"High-High (hotspot)": "#C44E52", "Low-Low (coldspot)": "#4C72B0", "Low-High": "#8DA0CB", "High-Low": "#FFD92F", "Not significant": "#D9D9D9"}
    moran_gdf.plot(color=moran_gdf["lisa_quadrant"].map(colors), edgecolor="white", linewidth=0.3, ax=ax)
    ax.set_axis_off()
    ax.set_title(f"LISA clusters (Queen, 999 perm), Moran's I={moran.I:.3f}, p={moran.p_sim:.3f}")
    handles = [plt.Rectangle((0,0),1,1, color=c) for c in colors.values()]
    ax.legend(handles, colors.keys(), loc="lower left", fontsize=8)
    plt.tight_layout()
    save_fig("B7", "lisa_cluster_map")
    plt.show()

run_step("B7", _b7)

### B8 — Venue concentration

In [ ]:
def _b8():
    conc = const_agg[["constituency", "state", "pct_pools_with_racialised", "top_pool_share_of_racialised"]].sort_values("top_pool_share_of_racialised", ascending=False)
    save_csv("B8", "venue_concentration", conc)
    print(conc.head(15))

    top20 = conc.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.barh(top20["constituency"] + " (" + top20["state"] + ")", top20["top_pool_share_of_racialised"], color="#8172B2")
    ax.set_xlabel("top pool's share of constituency's racialised reviews")
    ax.set_title("Top 20 constituencies by venue concentration")
    plt.tight_layout()
    save_fig("B8", "venue_concentration_top20")
    plt.show()

    dominant_idx = pool_racialised.groupby("constituency_code")["pool_racialised_count"].idxmax()
    pool_social = corpus.loc[corpus["social"] == "yes"].groupby("unique_identifier").size().rename("dominant_social_reviews")
    dominant_pools = pool_racialised.loc[dominant_idx].set_index("constituency_code").join(pool_social, on="unique_identifier")

    reliable_const = const_agg.loc[const_agg["reliable_n"], ["constituency_code", "constituency", "state", "social_reviews", "racialised_reviews"]].set_index("constituency_code")
    dp = dominant_pools.join(reliable_const, how="inner")
    dp["rest_racialised"] = dp["racialised_reviews"] - dp["pool_racialised_count"]
    dp["rest_social"] = dp["social_reviews"] - dp["dominant_social_reviews"]
    dp = dp.loc[dp["rest_social"] > 0].copy()
    dp["dominant_rate"] = dp["pool_racialised_count"] / dp["dominant_social_reviews"]
    dp["rest_rate"] = dp["rest_racialised"] / dp["rest_social"]
    dp["volume_share"] = dp["dominant_social_reviews"] / dp["social_reviews"]

    def two_prop_p(row):
        return proportions_ztest([row["pool_racialised_count"], row["rest_racialised"]], [row["dominant_social_reviews"], row["rest_social"]])[1]

    dp["p_value"] = dp.apply(two_prop_p, axis=1)
    dp["significantly_elevated"] = (dp["p_value"] < 0.05) & (dp["dominant_rate"] > dp["rest_rate"])
    save_csv("B8", "dominant_pool_vs_rest", dp[["constituency", "state", "dominant_rate", "rest_rate", "volume_share", "p_value", "significantly_elevated"]])
    print(f"dominant pools significantly elevated vs. rest of own constituency: {dp['significantly_elevated'].sum()} / {len(dp)}")
    RESULTS["B8_dominant_sig_elevated"] = f"{int(dp['significantly_elevated'].sum())}/{len(dp)}"

    fig, ax = plt.subplots(figsize=(7, 7))
    colors = np.where(dp["significantly_elevated"], "#C44E52", "#4C72B0")
    ax.scatter(dp["rest_rate"], dp["dominant_rate"], c=colors, alpha=0.7, s=dp["volume_share"] * 200 + 15)
    lims = [0, max(dp["rest_rate"].max(), dp["dominant_rate"].max()) * 1.05]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("rest-of-constituency rate"); ax.set_ylabel("dominant pool's rate")
    ax.set_title("Dominant pool vs. rest of constituency (red = significantly elevated, size = volume share)")
    plt.tight_layout()
    save_fig("B8", "dominant_pool_vs_rest")
    plt.show()

run_step("B8", _b8)

### A6 — Mean review year per constituency (confound check)

In [ ]:
def _a6():
    yr_stats = social_df[social_df["review_year"].between(2016, 2025)].groupby("constituency_code")["review_year"].agg(mean_review_year="mean", median_review_year="median")
    a6_df = analysis_df.set_index("constituency_code").join(yr_stats)
    rho, pval = stats.spearmanr(a6_df["mean_review_year"], a6_df["racialised_rate"])
    print(f"mean review year x racialised_rate: rho={rho:.4f}, p={pval:.4e}")
    save_csv("A6", "mean_review_year_per_constituency", a6_df[["mean_review_year", "median_review_year", "racialised_rate"]])
    RESULTS["A6_rho_mean_review_year"] = round(rho, 4)
    RESULTS["A6_p_mean_review_year"] = pval

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(a6_df["mean_review_year"], a6_df["racialised_rate"], alpha=0.6)
    ax.set_xlabel("mean review year")
    ax.set_ylabel("racialised_rate")
    plt.tight_layout()
    save_fig("A6", "mean_review_year_vs_rate")
    plt.show()

    if abs(rho) > 0.2:
        df2 = analysis_df.set_index("constituency_code").join(yr_stats).reset_index()
        df2["mean_review_year_c"] = df2["mean_review_year"] - df2["mean_review_year"].mean()
        formula_plus = FULL_FORMULA + " + mean_review_year_c"
        m_base = smf.glm(FULL_FORMULA, data=df2, family=sm.families.NegativeBinomial(), offset=np.log(df2["social_reviews"])).fit()
        m_plus = smf.glm(formula_plus, data=df2, family=sm.families.NegativeBinomial(), offset=np.log(df2["social_reviews"])).fit()
        comparison = pd.DataFrame({"base_coef": m_base.params, "base_p": m_base.pvalues}).join(
            pd.DataFrame({"plus_year_coef": m_plus.params, "plus_year_p": m_plus.pvalues}), how="outer")
        save_csv("A6", "model_with_mean_review_year", comparison)
        print(comparison.round(4))
    else:
        print(f"|rho|={abs(rho):.3f} <= 0.2 -- refit not triggered")

run_step("A6", _a6)

### D2 — Mean-variance relationship

In [ ]:
def _d2():
    df = analysis_df.copy()
    offset = np.log(df["social_reviews"])
    p = smf.glm("racialised_reviews ~ 1", data=df, family=sm.families.Poisson(), offset=offset).fit()
    df["fitted_mu"] = p.mu

    df_sorted = df.sort_values("fitted_mu").reset_index(drop=True)
    n_bins = 15
    df_sorted["bin"] = pd.qcut(df_sorted.index, n_bins, labels=False)
    binned = df_sorted.groupby("bin").agg(
        mean_fitted=("fitted_mu", "mean"),
        obs_mean=("racialised_reviews", "mean"),
        obs_var=("racialised_reviews", "var"),
        n=("racialised_reviews", "size"),
    )
    save_csv("D2", "mean_variance_bins", binned)
    print(binned.round(3))

    disp_null = p.pearson_chi2 / p.df_resid  # phi for the quasi-Poisson reference curve
    mu_all = df["fitted_mu"]
    aux = sm.OLS(((df["racialised_reviews"] - mu_all) ** 2 - mu_all) / mu_all, mu_all).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(binned["obs_mean"], binned["obs_var"], color="black", zorder=3, label="observed (binned)")
    mgrid = np.linspace(binned["obs_mean"].min(), binned["obs_mean"].max(), 100)
    ax.plot(mgrid, mgrid, label="var = mean (Poisson)", color="steelblue")
    ax.plot(mgrid, disp_null * mgrid, label=f"var = phi*mean (quasi-Poisson, phi={disp_null:.2f})", color="seagreen")
    ax.plot(mgrid, mgrid + alpha_hat * mgrid ** 2, label=f"var = mean + alpha*mean^2 (NB, alpha={alpha_hat:.4f})", color="firebrick")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("mean (log scale)"); ax.set_ylabel("variance (log scale)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig("D2", "mean_variance_relationship")
    plt.show()

    log_mean = np.log(binned["obs_mean"])
    log_var = np.log(binned["obs_var"])
    slope_fit = sm.OLS(log_var, sm.add_constant(log_mean)).fit()
    slope = slope_fit.params.iloc[1]
    print(f"\nlog(variance) ~ log(mean) slope: {slope:.3f}")
    RESULTS["D2_mean_variance_slope"] = round(slope, 3)

run_step("D2", _d2)

### R1 -- Linear regression baseline (reference only, NOT a candidate model)

Pedagogical comparison showing why OLS is inadequate for a bounded proportion outcome (`rate_social`). Does not feed model selection; no later section cites its coefficients as findings.

In [ ]:
def _r1():
    # REFERENCE ONLY -- not a candidate model. Shows why OLS is inadequate for this outcome.
    # Nothing here feeds model selection; no later section may cite these coefficients as findings.
    from statsmodels.iolib.summary2 import summary_col
    from statsmodels.stats.diagnostic import het_breuschpagan
    from esda.moran import Moran

    preds = FULL_FORMULA.split("~")[1].strip()
    df = analysis_df.copy()
    df["rate_social"] = df["racialised_reviews"] / df["social_reviews"]
    df["non_racialised_social"] = df["social_reviews"] - df["racialised_reviews"]

    ols = smf.ols(f"rate_social ~ {preds}", data=df).fit()
    wls = smf.wls(f"rate_social ~ {preds}", data=df, weights=df["social_reviews"]).fit()
    glm_formula = f"racialised_reviews + non_racialised_social ~ {preds}"
    glm = smf.glm(glm_formula, data=df, family=sm.families.Binomial()).fit()
    glm_hc1 = smf.glm(glm_formula, data=df, family=sm.families.Binomial()).fit(cov_type="HC1")

    table = summary_col([ols, wls, glm, glm_hc1], stars=True,
                         model_names=["OLS", "WLS", "Binomial GLM", "Binomial GLM (HC1)"])
    print(table)
    (OUT_DIR / "R1_summary_col_baseline.txt").write_text(str(table))
    print(f"saved {OUT_DIR / 'R1_summary_col_baseline.txt'}")

    # --- OLS fitted values outside [0,1] ---
    df["ols_fitted"] = ols.fittedvalues
    oob = df[(df["ols_fitted"] < 0) | (df["ols_fitted"] > 1)]
    print(f"\nOLS fitted values outside [0,1]: {len(oob)} of {len(df)} constituencies")
    print(oob[["constituency_code", "constituency", "ols_fitted"]])
    print("-- OLS on a bounded proportion has no floor/ceiling, so it can and does predict impossible rates.")

    # --- residuals vs fitted: OLS and GLM deviance, side by side ---
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].scatter(ols.fittedvalues, ols.resid, alpha=0.6)
    axes[0].axhline(0, color="grey", lw=1)
    axes[0].set_xlabel("OLS fitted"); axes[0].set_ylabel("OLS residual")
    axes[1].scatter(glm.fittedvalues, glm.resid_deviance, alpha=0.6, color="firebrick")
    axes[1].axhline(0, color="grey", lw=1)
    axes[1].set_xlabel("GLM fitted (probability)"); axes[1].set_ylabel("GLM deviance residual")
    plt.tight_layout()
    save_fig("R1", "resid_vs_fitted_ols_glm")
    plt.show()

    # --- OLS residuals vs n_social (funnel check) ---
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(df["social_reviews"], ols.resid, alpha=0.6)
    ax.axhline(0, color="grey", lw=1)
    ax.set_xlabel("social_reviews (denominator)"); ax.set_ylabel("OLS residual")
    plt.tight_layout()
    save_fig("R1", "ols_resid_vs_n_social")
    plt.show()
    corr_abs = np.corrcoef(df["social_reviews"], np.abs(ols.resid))[0, 1]
    funnel = "a funnel is visible" if abs(corr_abs) > 0.2 else "no clear funnel is visible"
    print(f"corr(|OLS resid|, social_reviews) = {corr_abs:.3f} -- {funnel}: residual spread "
          f"{'shrinks' if corr_abs < 0 else 'does not shrink'} as the denominator grows, "
          f"the signature of ignoring unequal counts.")

    # --- Breusch-Pagan ---
    bp_stat, bp_p, _, _ = het_breuschpagan(ols.resid, ols.model.exog)
    print(f"\nBreusch-Pagan: LM={bp_stat:.3f}, p={bp_p:.4f} -- "
          f"{'heteroskedasticity detected' if bp_p < 0.05 else 'no evidence of heteroskedasticity'} in the OLS residuals.")

    # --- QQ plot ---
    fig, ax = plt.subplots(figsize=(5, 5))
    sm.qqplot(ols.resid, line="s", ax=ax)
    plt.tight_layout()
    save_fig("R1", "ols_resid_qq")
    plt.show()

    # --- Moran's I on OLS and GLM deviance residuals, existing spatial weights `w` ---
    resid_df = df[["constituency_code"]].copy()
    resid_df["ols_resid"] = ols.resid.values
    resid_df["glm_dev_resid"] = glm.resid_deviance
    moran_input = moran_gdf[["constituency_code"]].merge(resid_df, on="constituency_code", how="left")
    assert moran_input["constituency_code"].tolist() == moran_gdf["constituency_code"].tolist(), \
        "residuals must be reindexed to moran_gdf/w row order before Moran's I"

    moran_ols = Moran(moran_input["ols_resid"].values, w, permutations=999)
    moran_glm = Moran(moran_input["glm_dev_resid"].values, w, permutations=999)
    print(f"\nMoran's I (OLS resid) = {moran_ols.I:.4f}, p = {moran_ols.p_sim:.4f} -- "
          f"{'spatially clustered' if moran_ols.p_sim < 0.05 else 'no significant spatial clustering'} residuals.")
    print(f"Moran's I (GLM deviance resid) = {moran_glm.I:.4f}, p = {moran_glm.p_sim:.4f} -- "
          f"{'spatially clustered' if moran_glm.p_sim < 0.05 else 'no significant spatial clustering'} residuals.")

    # --- GLM dispersion ---
    disp = glm.pearson_chi2 / glm.df_resid
    print(f"\nGLM dispersion (Pearson chi2/df_resid) = {disp:.3f} -- "
          f"{'overdispersed relative to binomial variance' if disp > 1.5 else 'roughly consistent with binomial variance'}.")

    # --- largest OLS -> WLS coefficient change ---
    common = [t for t in ols.params.index if t in wls.params.index]
    pct_change = ((wls.params[common] - ols.params[common]) / ols.params[common]).abs() * 100
    biggest = pct_change.idxmax()
    print(f"\nLargest OLS->WLS coefficient change: {biggest}, {pct_change[biggest]:.1f}% of the OLS coefficient "
          f"(OLS={ols.params[biggest]:.5f}, WLS={wls.params[biggest]:.5f}) -- "
          f"this isolates the cost of ignoring unequal denominators (social_reviews) across constituencies.")

run_step("R1", _r1)


### D3 — Fit all three models (Poisson, quasi-Poisson, negative binomial)

In [ ]:
def _d3():
    global alpha_hat_d3
    df = analysis_df.copy()
    offset = np.log(df["social_reviews"])

    M1 = smf.glm(FULL_FORMULA, data=df, family=sm.families.Poisson(), offset=offset).fit()
    disp_m1 = M1.pearson_chi2 / M1.df_resid

    M2 = smf.glm(FULL_FORMULA, data=df, family=sm.families.Poisson(), offset=offset).fit(scale="X2")

    mu = M1.mu
    aux = sm.OLS(((df["racialised_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat_d3 = max(float(aux.params.iloc[0]), 1e-6)
    M3 = smf.glm(FULL_FORMULA, data=df, family=sm.families.NegativeBinomial(alpha=alpha_hat_d3), offset=offset).fit()
    disp_m3 = M3.pearson_chi2 / M3.df_resid

    def rr(model, term):
        return np.exp(model.params[term])

    rows = {
        "dispersion (Pearson chi2/df)": [disp_m1, disp_m1, disp_m3],
        "phi or alpha": [np.nan, disp_m1, alpha_hat_d3],
        "log-likelihood": [M1.llf, np.nan, M3.llf],
        "AIC": [M1.aic, np.nan, M3.aic],
        "coef: foreign pop (rate ratio)": [rr(M1, "foreign_population_pct_c"), rr(M2, "foreign_population_pct_c"), rr(M3, "foreign_population_pct_c")],
        "coef: density (rate ratio)": [rr(M1, "log_population_density_c"), rr(M2, "log_population_density_c"), rr(M3, "log_population_density_c")],
        "coef: benefits (rate ratio)": [rr(M1, "sgb_ii_recipients_per_1k_c"), rr(M2, "sgb_ii_recipients_per_1k_c"), rr(M3, "sgb_ii_recipients_per_1k_c")],
        "SE: foreign pop": [M1.bse["foreign_population_pct_c"], M2.bse["foreign_population_pct_c"], M3.bse["foreign_population_pct_c"]],
        "SE: density": [M1.bse["log_population_density_c"], M2.bse["log_population_density_c"], M3.bse["log_population_density_c"]],
        "SE: benefits": [M1.bse["sgb_ii_recipients_per_1k_c"], M2.bse["sgb_ii_recipients_per_1k_c"], M3.bse["sgb_ii_recipients_per_1k_c"]],
        "n significant at p<0.05": [int((M1.pvalues < 0.05).sum()), int((M2.pvalues < 0.05).sum()), int((M3.pvalues < 0.05).sum())],
    }
    table = pd.DataFrame(rows, index=["Poisson", "quasi-Poisson", "neg. binomial"]).T
    save_csv("D3", "three_model_comparison", table)
    print(table.round(4))
    print("\nnote: quasi-Poisson has no likelihood -- AIC and LR tests are unavailable for it; "
          "model comparison elsewhere in this notebook uses negative binomial.")

run_step("D3", _d3)

### D4 — Deviance tables (analysis of deviance)

In [ ]:
def _d4():
    df = analysis_df.copy()
    offset = np.log(df["social_reviews"])
    order = ["foreign_population_pct_c", "log_population_density_c", "sgb_ii_recipients_per_1k_c", "is_east", "in_split_city"]

    def deviance_table(family):
        rows = []
        terms = []
        m = smf.glm("racialised_reviews ~ 1", data=df, family=family, offset=offset).fit()
        rows.append({"step": "intercept only", "deviance": m.deviance, "df_resid": m.df_resid, "delta_deviance": np.nan, "delta_df": np.nan, "p": np.nan})
        prev_dev, prev_df = m.deviance, m.df_resid
        for term in order:
            terms.append(term)
            formula = "racialised_reviews ~ " + " + ".join(terms)
            m = smf.glm(formula, data=df, family=family, offset=offset).fit()
            delta_dev = prev_dev - m.deviance
            delta_df = prev_df - m.df_resid
            p = stats.chi2.sf(delta_dev, delta_df) if delta_df > 0 else np.nan
            rows.append({"step": f"+ {term}", "deviance": m.deviance, "df_resid": m.df_resid, "delta_deviance": delta_dev, "delta_df": delta_df, "p": p})
            prev_dev, prev_df = m.deviance, m.df_resid
        return pd.DataFrame(rows)

    poisson_table = deviance_table(sm.families.Poisson())
    nb_table = deviance_table(sm.families.NegativeBinomial(alpha=alpha_hat_d3))

    save_csv("D4", "deviance_table_poisson", poisson_table)
    save_csv("D4", "deviance_table_negbinomial", nb_table)
    print("--- Poisson analysis-of-deviance ---"); print(poisson_table.round(4))
    print("\n--- Negative binomial analysis-of-deviance ---"); print(nb_table.round(4))
    last_dev, last_df = poisson_table["deviance"].iloc[-1], poisson_table["df_resid"].iloc[-1]
    print(f"\nPoisson residual deviance ({last_dev:.1f}) vs residual df ({last_df}): ratio {last_dev / last_df:.2f}")

run_step("D4", _d4)

### B9 — The chosen model

In [ ]:
def _b9():
    global nb_main, df9
    df9 = analysis_df.copy()
    offset = np.log(df9["social_reviews"])

    p = smf.glm(FULL_FORMULA, data=df9, family=sm.families.Poisson(), offset=offset).fit()
    mu = p.mu
    aux = sm.OLS(((df9["racialised_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
    fam = sm.families.NegativeBinomial(alpha=alpha_hat)

    nb_main = smf.glm(FULL_FORMULA, data=df9, family=fam, offset=offset).fit(cov_type="cluster", cov_kwds={"groups": df9["state"]})
    res = pd.DataFrame({"rate_ratio": np.exp(nb_main.params).round(3), "ci_low": np.exp(nb_main.conf_int()[0]).round(3),
                         "ci_high": np.exp(nb_main.conf_int()[1]).round(3), "p": nb_main.pvalues.round(4)})
    save_csv("B9", "main_model_rate_ratios", res)
    print("--- main model ---")
    print(res)

    df9["second_vote_afd_pct_c"] = df9["second_vote_afd_pct"] - df9["second_vote_afd_pct"].mean()
    formula_afd = FULL_FORMULA + " + second_vote_afd_pct_c"
    nb_afd = smf.glm(formula_afd, data=df9, family=fam, offset=offset).fit(cov_type="cluster", cov_kwds={"groups": df9["state"]})
    res_afd = pd.DataFrame({"rate_ratio": np.exp(nb_afd.params).round(3), "ci_low": np.exp(nb_afd.conf_int()[0]).round(3),
                             "ci_high": np.exp(nb_afd.conf_int()[1]).round(3), "p": nb_afd.pvalues.round(4)})
    save_csv("B9", "model_with_afd", res_afd)
    print("--- model + AfD second-vote share ---")
    print(res_afd)

    bivariate = []
    for label, sub in [("West", df9[df9["is_east"] == 0]), ("East", df9[df9["is_east"] == 1])]:
        rho, pval = stats.spearmanr(sub["second_vote_afd_pct"], sub["racialised_rate"])
        bivariate.append({"region": label, "n": len(sub), "rho": rho, "p": pval})
    bivariate_df = pd.DataFrame(bivariate)
    save_csv("B9", "afd_bivariate_by_region", bivariate_df)
    print(bivariate_df)

    fig, ax = plt.subplots(figsize=(6, 5))
    for label, sub, c in [("West", df9[df9["is_east"] == 0], "steelblue"), ("East", df9[df9["is_east"] == 1], "firebrick")]:
        ax.scatter(sub["second_vote_afd_pct"], sub["racialised_rate"], alpha=0.6, label=label, color=c)
    ax.set_xlabel("second_vote_afd_pct")
    ax.set_ylabel("racialised_rate")
    ax.legend()
    plt.tight_layout()
    save_fig("B9", "afd_bivariate_east_west")
    plt.show()

    for term in nb_main.params.index:
        RESULTS[f"B9_rr_{term}"] = round(float(np.exp(nb_main.params[term])), 3)
        RESULTS[f"B9_p_{term}"] = round(float(nb_main.pvalues[term]), 4)

run_step("B9", _b9)

## Task B -- German vs. non-German split

Reuses the final model's predictor set and specification (NB GLM, log-offset, cluster SEs by state).

### B1 -- Feasibility gate (German vs. non-German split)

Run first; the split-model analysis below only proceeds if this gate passes.

In [ ]:
def _bsplit1():
    global df_split
    GROUP5 = {"german": ["Germany"], "nongerman": ["Arab", "Türkiye", "Other"], "unknown": ["Unknown"]}

    counts = {}
    for grp, vals in GROUP5.items():
        soc_sub = social_df[social_df["ethnicity_group5"].isin(vals)]
        rac_sub = racialised_df[racialised_df["ethnicity_group5"].isin(vals)]
        counts[f"n_soc_{grp}"] = soc_sub.groupby("constituency_code").size()
        counts[f"n_rac_{grp}"] = rac_sub.groupby("constituency_code").size()

    df_split = analysis_df.copy()
    for col, s in counts.items():
        df_split[col] = df_split["constituency_code"].map(s).fillna(0).astype(int)

    cols = ["n_rac_german", "n_soc_german", "n_rac_nongerman", "n_soc_nongerman", "n_rac_unknown", "n_soc_unknown"]
    desc = df_split[cols].describe(percentiles=[.05, .25, .5, .75, .95])
    save_csv("BSPLIT1", "feasibility_describe", desc)
    print(desc.round(2))

    zero_german = int((df_split["n_rac_german"] == 0).sum())
    zero_nongerman = int((df_split["n_rac_nongerman"] == 0).sum())
    print(f"\nzero n_rac_german: {zero_german}/{len(df_split)} ({zero_german/len(df_split):.1%})")
    print(f"zero n_rac_nongerman: {zero_nongerman}/{len(df_split)} ({zero_nongerman/len(df_split):.1%})")

    median_nongerman = df_split["n_rac_nongerman"].median()
    gate_fail = (median_nongerman < 5) or (zero_german / len(df_split) > 1/3) or (zero_nongerman / len(df_split) > 1/3)
    verdict = "FAIL" if gate_fail else "PASS"
    print(f"\nmedian n_rac_nongerman = {median_nongerman:.1f}")
    print(f"GATE: {verdict}")
    RESULTS["BSPLIT1_gate"] = verdict
    RESULTS["BSPLIT1_median_n_rac_nongerman"] = float(median_nongerman)
    return verdict

gate_verdict = run_step("BSPLIT1", _bsplit1)

### B2 -- Two denominators (shared vs. group-specific)

In [ ]:
def _bsplit2():
    preds = FULL_FORMULA.split("~")[1].strip()

    def fit_nb_offset(df, endog_col, offset_col):
        d = df.copy()
        offset = np.log(d[offset_col].replace(0, np.nan))
        d = d.loc[offset.notna()]
        offset = offset.loc[offset.notna()]
        p = smf.glm(f"{endog_col} ~ {preds}", data=d, family=sm.families.Poisson(), offset=offset).fit()
        mu = p.mu
        aux = sm.OLS(((d[endog_col] - mu) ** 2 - mu) / mu, mu).fit()
        alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
        fam = sm.families.NegativeBinomial(alpha=alpha_hat)
        return smf.glm(f"{endog_col} ~ {preds}", data=d, family=fam, offset=offset).fit(
            cov_type="cluster", cov_kwds={"groups": d["state"]})

    global m_shared_german, m_shared_nongerman, m_specific_german, m_specific_nongerman
    m_shared_german = fit_nb_offset(df_split, "n_rac_german", "social_reviews")
    m_shared_nongerman = fit_nb_offset(df_split, "n_rac_nongerman", "social_reviews")
    m_specific_german = fit_nb_offset(df_split, "n_rac_german", "n_soc_german")
    m_specific_nongerman = fit_nb_offset(df_split, "n_rac_nongerman", "n_soc_nongerman")

    for label, m in [("shared/German", m_shared_german), ("shared/non-German", m_shared_nongerman),
                     ("group-specific/German", m_specific_german), ("group-specific/non-German", m_specific_nongerman)]:
        res = pd.DataFrame({"rate_ratio": np.exp(m.params).round(3), "p": m.pvalues.round(4)})
        save_csv("BSPLIT2", f"model_{label.replace('/', '_')}", res)
        print(f"--- {label} ---")
        print(res)

run_step("BSPLIT2", _bsplit2)

### B3 -- Stacked model with interactions (headline test of Task B)

In [ ]:
def build_long(df, rac_cols_by_group, soc_cols_by_group):
    preds_cols = ["constituency_code", "state", "foreign_population_pct_c", "log_population_density_c",
                  "sgb_ii_recipients_per_1k_c", "is_east", "in_split_city"]
    parts = []
    for grp in rac_cols_by_group:
        tmp = df[preds_cols + [rac_cols_by_group[grp], soc_cols_by_group[grp]]].copy()
        tmp = tmp.rename(columns={rac_cols_by_group[grp]: "n_rac", soc_cols_by_group[grp]: "n_soc"})
        tmp["group"] = grp
        parts.append(tmp)
    long_df = pd.concat(parts, ignore_index=True)
    long_df["n_nonrac"] = long_df["n_soc"] - long_df["n_rac"]
    long_df["group"] = pd.Categorical(long_df["group"], categories=list(rac_cols_by_group.keys()))
    return long_df

def fit_stacked(long_df):
    preds = FULL_FORMULA.split("~")[1].strip()
    formula = f"n_rac + n_nonrac ~ group * ({preds})"
    fit = smf.glm(formula, data=long_df, family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": long_df["constituency_code"]})
    interaction_terms = [t for t in fit.params.index if ":" in t]
    R = np.zeros((len(interaction_terms), len(fit.params)))
    for i, t in enumerate(interaction_terms):
        R[i, fit.params.index.get_loc(t)] = 1
    wald = fit.wald_test(R, scalar=True)
    return fit, interaction_terms, wald

def moran_by_group(long_df, fit):
    from esda.moran import Moran
    long_df = long_df.copy()
    long_df["dev_resid"] = fit.resid_deviance
    out = {}
    for grp in long_df["group"].cat.categories:
        sub = long_df[long_df["group"] == grp]
        merged = moran_gdf[["constituency_code"]].merge(sub[["constituency_code", "dev_resid"]], on="constituency_code", how="left")
        assert merged["constituency_code"].tolist() == moran_gdf["constituency_code"].tolist()
        m = Moran(merged["dev_resid"].values, w, permutations=999)
        out[grp] = (m.I, m.p_sim)
        print(f"Moran's I ({grp} deviance resid) = {m.I:.4f}, p = {m.p_sim:.4f}")
    return out

def _bsplit3():
    global long_df_specific, fit_specific, interaction_terms_specific, wald_specific
    long_df_specific = build_long(df_split,
                                   {"german": "n_rac_german", "nongerman": "n_rac_nongerman"},
                                   {"german": "n_soc_german", "nongerman": "n_soc_nongerman"})
    fit_specific, interaction_terms_specific, wald_specific = fit_stacked(long_df_specific)

    res = pd.DataFrame({"coef": fit_specific.params, "ci_low": fit_specific.conf_int()[0],
                         "ci_high": fit_specific.conf_int()[1], "p": fit_specific.pvalues})
    save_csv("BSPLIT3", "stacked_group_specific_denom", res)
    print(res.loc[interaction_terms_specific].round(4))
    print(f"\nJoint Wald test on all {len(interaction_terms_specific)} interaction terms: "
          f"chi2={wald_specific.statistic:.3f}, df={len(interaction_terms_specific)}, p={wald_specific.pvalue:.4f}")
    print("-- this is the headline test of Task B: whether predictor slopes differ by reviewer group.")

    moran_by_group(long_df_specific, fit_specific)

run_step("BSPLIT3", _bsplit3)

### B4 -- Unknown sensitivity

In [ ]:
def _bsplit4():
    df_split["n_rac_nongerman_plus_unk"] = df_split["n_rac_nongerman"] + df_split["n_rac_unknown"]
    df_split["n_soc_nongerman_plus_unk"] = df_split["n_soc_nongerman"] + df_split["n_soc_unknown"]

    long_folded = build_long(df_split,
                              {"german": "n_rac_german", "nongerman": "n_rac_nongerman_plus_unk"},
                              {"german": "n_soc_german", "nongerman": "n_soc_nongerman_plus_unk"})
    _, terms_folded, wald_folded = fit_stacked(long_folded)
    print(f"Unknown folded into non-German: joint Wald p = {wald_folded.pvalue:.4f} "
          f"({'same conclusion' if (wald_folded.pvalue < 0.05) == (wald_specific.pvalue < 0.05) else 'CONCLUSION CHANGES'} vs. headline p={wald_specific.pvalue:.4f})")

    long_3grp = build_long(df_split,
                            {"german": "n_rac_german", "nongerman": "n_rac_nongerman", "unknown": "n_rac_unknown"},
                            {"german": "n_soc_german", "nongerman": "n_soc_nongerman", "unknown": "n_soc_unknown"})
    _, terms_3grp, wald_3grp = fit_stacked(long_3grp)
    print(f"Unknown as third group level: joint Wald p = {wald_3grp.pvalue:.4f} "
          f"({'same conclusion' if (wald_3grp.pvalue < 0.05) == (wald_specific.pvalue < 0.05) else 'CONCLUSION CHANGES'} vs. headline p={wald_specific.pvalue:.4f})")

    RESULTS["BSPLIT4_wald_p_folded"] = round(float(wald_folded.pvalue), 4)
    RESULTS["BSPLIT4_wald_p_3level"] = round(float(wald_3grp.pvalue), 4)

run_step("BSPLIT4", _bsplit4)

### B5 -- Output: comparison table and summary

In [ ]:
def _bsplit5():
    preds = ["foreign_population_pct_c", "log_population_density_c", "sgb_ii_recipients_per_1k_c", "is_east", "in_split_city"]

    def comparison_table(fit, german_model, nongerman_model):
        rows = []
        for p in preds:
            inter_term = f"group[T.nongerman]:{p}"
            rows.append({
                "predictor": p,
                "german_coef": german_model.params.get(p, np.nan),
                "nongerman_coef": nongerman_model.params.get(p, np.nan),
                "interaction_coef": fit.params.get(inter_term, np.nan),
                "interaction_p": fit.pvalues.get(inter_term, np.nan),
            })
        return pd.DataFrame(rows)

    long_df_shared = build_long(df_split, {"german": "n_rac_german", "nongerman": "n_rac_nongerman"},
                                 {"german": "social_reviews", "nongerman": "social_reviews"})
    fit_shared, terms_shared, wald_shared = fit_stacked(long_df_shared)

    table_specific = comparison_table(fit_specific, m_specific_german, m_specific_nongerman)
    table_shared = comparison_table(fit_shared, m_shared_german, m_shared_nongerman)

    save_csv("BSPLIT5", "comparison_group_specific_denom", table_specific)
    save_csv("BSPLIT5", "comparison_shared_denom", table_shared)
    print("=== group-specific denominator (primary for directionality) ===")
    print(table_specific.round(4))
    print(f"joint Wald p (group-specific) = {wald_specific.pvalue:.4f}")
    print("\n=== shared denominator (n_soc_total for both groups) ===")
    print(table_shared.round(4))
    print(f"joint Wald p (shared) = {wald_shared.pvalue:.4f}")

    differ = wald_specific.pvalue < 0.05
    survives = (wald_specific.pvalue < 0.05) == (RESULTS.get("BSPLIT4_wald_p_folded", 1) < 0.05) and \
               (wald_specific.pvalue < 0.05) == (RESULTS.get("BSPLIT4_wald_p_3level", 1) < 0.05)
    print(f"\nSummary: predictor slopes {'differ' if differ else 'do not differ'} significantly between German and "
          f"non-German reviewers (joint Wald p={wald_specific.pvalue:.4f}). This conclusion "
          f"{'survives' if survives else 'does NOT survive'} the Unknown-handling sensitivity checks (B4). "
          f"{'The shared-denominator version tells a similar story' if (wald_shared.pvalue < 0.05) == differ else 'The shared-denominator version differs, consistent with reviewer-pool composition confounding the shared-denominator comparison'}.")

run_step("BSPLIT5", _bsplit5)

### D6 — Residuals, observed vs. predicted

In [ ]:
def _d6():
    pred_rate = np.exp(nb_main.fittedvalues) / df9["social_reviews"]
    obs_rate = df9["racialised_rate"]
    rho_r, p_r = stats.spearmanr(obs_rate, pred_rate)
    pear_r, p_pr = stats.pearsonr(obs_rate, pred_rate)
    print(f"observed vs predicted rate: spearman rho={rho_r:.4f} (p={p_r:.2e}), pearson r={pear_r:.4f} (p={p_pr:.2e})")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(obs_rate, pred_rate, alpha=0.6)
    lims = [0, max(obs_rate.max(), pred_rate.max()) * 1.05]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("observed rate"); ax.set_ylabel("predicted rate")
    plt.tight_layout(); save_fig("D6", "observed_vs_predicted_rate"); plt.show()

    pred_count = np.exp(nb_main.fittedvalues)
    obs_count = df9["racialised_reviews"]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(obs_count, pred_count, alpha=0.6)
    ax.set_xscale("log"); ax.set_yscale("log")
    lims = [max(obs_count.min(), 1), max(obs_count.max(), pred_count.max()) * 1.05]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("observed count (log)"); ax.set_ylabel("predicted count (log)")
    plt.tight_layout(); save_fig("D6", "observed_vs_predicted_count"); plt.show()

    resid = nb_main.resid_pearson
    fitted = nb_main.fittedvalues

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].scatter(fitted, resid, alpha=0.6)
    axes[0].axhline(0, color="k", lw=1)
    axes[0].set_xlabel("fitted values"); axes[0].set_ylabel("Pearson residuals"); axes[0].set_title("residuals vs fitted")

    axes[1].scatter(df9["social_reviews"], resid, alpha=0.6)
    axes[1].set_xscale("log")
    axes[1].axhline(0, color="k", lw=1)
    axes[1].set_xlabel("social_reviews (log)"); axes[1].set_ylabel("Pearson residuals"); axes[1].set_title("residuals vs offset variable")

    axes[2].hist(resid, bins=25, color="steelblue", edgecolor="black")
    axes[2].set_title("residual histogram")
    plt.tight_layout(); save_fig("D6", "residuals_fitted_offset_hist"); plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, pred in zip(axes, ["foreign_population_pct_c", "log_population_density_c", "sgb_ii_recipients_per_1k_c"]):
        ax.scatter(df9[pred], resid, alpha=0.6)
        ax.axhline(0, color="k", lw=1)
        ax.set_xlabel(pred); ax.set_ylabel("Pearson residuals")
    plt.tight_layout(); save_fig("D6", "residuals_vs_predictors"); plt.show()

    fig, ax = plt.subplots(figsize=(6, 6))
    sm.qqplot(resid, line="45", ax=ax)
    plt.tight_layout(); save_fig("D6", "residuals_qqplot"); plt.show()

    plot_df = df9.assign(resid_pearson=resid.values)
    state_order = plot_df.groupby("state")["resid_pearson"].median().sort_values().index
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=plot_df, x="state", y="resid_pearson", order=state_order, ax=ax, color="lightsteelblue")
    ax.tick_params(axis="x", rotation=75)
    ax.axhline(0, color="k", lw=1)
    plt.tight_layout(); save_fig("D6", "residuals_by_state"); plt.show()

    from esda.moran import Moran
    resid_gdf = moran_gdf.merge(df9[["constituency_code"]].assign(resid_pearson=resid.values), on="constituency_code", how="left")
    moran_resid = Moran(resid_gdf["resid_pearson"].fillna(0).values, w, permutations=999)
    print(f"\nresidual Moran's I = {moran_resid.I:.4f}, p = {moran_resid.p_sim:.4f}")

    cooks = nb_main.get_influence().cooks_distance[0]
    cooks_df = pd.DataFrame({"constituency": df9["constituency"], "state": df9["state"], "cooks_d": cooks}).sort_values("cooks_d", ascending=False)
    save_csv("D6", "cooks_distance_top10", cooks_df.head(10))
    print("\n--- top 10 Cook's distance ---")
    print(cooks_df.head(10))

    RESULTS["D6_obs_pred_rate_spearman"] = round(rho_r, 4)
    RESULTS["D6_obs_pred_rate_pearson"] = round(pear_r, 4)
    RESULTS["D6_residual_moran_I"] = round(moran_resid.I, 4)

run_step("D6", _d6)

### B10 — VIF, robustness

In [ ]:
def _b10():
    from patsy import dmatrix
    X = dmatrix("foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east + in_split_city", data=df9, return_type="dataframe")
    vif = pd.DataFrame({"variable": X.columns, "vif": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
    save_csv("B10", "vif", vif)
    print("--- VIF ---")
    print(vif.round(2))

    def refit(subset, label, drop_terms=()):
        terms = ["foreign_population_pct_c", "log_population_density_c", "sgb_ii_recipients_per_1k_c", "is_east", "in_split_city"]
        terms = [t for t in terms if t not in drop_terms]
        f = "racialised_reviews ~ " + " + ".join(terms)
        m = smf.glm(f, data=subset, family=sm.families.NegativeBinomial(alpha=nb_main.model.family.alpha), offset=np.log(subset["social_reviews"])).fit(
            cov_type="cluster", cov_kwds={"groups": subset["state"]})
        return np.exp(m.params).round(3).rename(f"{label}_{len(subset)}")

    checks = pd.concat([
        np.exp(nb_main.params).round(3).rename("main_299"),
        refit(df9[df9["in_split_city"] == 0], "no_split", ["in_split_city"]),
        refit(df9[df9["racialised_reviews"] >= 10], "ten_plus"),
        refit(df9[df9["is_east"] == 0], "west_only", ["is_east"]),
        refit(df9[df9["state"] != "Nordrhein-Westfalen"], "no_nrw"),
    ], axis=1)
    save_csv("B10", "five_subset_robustness", checks)
    print("\n--- five-subset robustness (rate ratios) ---")
    print(checks)

run_step("B10", _b10)

### C2 — Robustness on the all-reviews denominator

In [ ]:
def _c2():
    df_c2 = analysis_df.copy()
    offset_all = np.log(df_c2["total_text_reviews"])
    p = smf.glm(FULL_FORMULA, data=df_c2, family=sm.families.Poisson(), offset=offset_all).fit()
    mu = p.mu
    aux = sm.OLS(((df_c2["racialised_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
    fam = sm.families.NegativeBinomial(alpha=alpha_hat)
    nb_all = smf.glm(FULL_FORMULA, data=df_c2, family=fam, offset=offset_all).fit(cov_type="cluster", cov_kwds={"groups": df_c2["state"]})

    res_all = pd.DataFrame({"rate_ratio": np.exp(nb_all.params).round(3), "p": nb_all.pvalues.round(4)})
    res_social = pd.DataFrame({"rate_ratio": np.exp(nb_main.params).round(3), "p": nb_main.pvalues.round(4)})
    side_by_side = res_social.join(res_all, lsuffix="_social_denom", rsuffix="_all_denom")
    save_csv("C2", "offset_comparison", side_by_side)
    print(side_by_side)

run_step("C2", _c2)

### C1 — Content by place

In [ ]:
def _c1():
    d = racialised_df.merge(moran_gdf[["constituency_code", "lisa_quadrant"]], on="constituency_code", how="left")
    d = d.merge(analysis_df[["constituency_code", "is_east", "foreign_population_pct"]], on="constituency_code", how="left")
    median_foreign = analysis_df["foreign_population_pct"].median()
    d["foreign_pop_split"] = np.where(d["foreign_population_pct"] >= median_foreign, "high", "low")
    d["lisa_group"] = d["lisa_quadrant"].map({"High-High (hotspot)": "hotspot", "Low-Low (coldspot)": "coldspot"}).fillna("neither")
    d["east_west"] = np.where(d["is_east"] == 1, "East", "West")

    for cut_col, cut_name in [("lisa_group", "lisa"), ("east_west", "east_west"), ("foreign_pop_split", "foreign_pop_median_split")]:
        for dim in ["theme", "context"]:
            exploded = d[[cut_col, dim]].explode(dim)
            ct = pd.crosstab(exploded[dim].to_numpy(), exploded[cut_col].to_numpy(), normalize="columns") * 100
            ct.index.name, ct.columns.name = dim, cut_col
            save_csv("C1", f"{dim}_by_{cut_name}", ct)
            print(f"--- {dim} composition by {cut_name} (col %) ---")
            print(ct.round(1))

run_step("C1", _c1)

## Observations

_(one line per interesting-but-out-of-scope note, added during the run)_

## Run summary — errors and RESULTS_SUMMARY.md

In [ ]:
print(f"{len(ERRORS)} cell(s) skipped due to errors:")
for cid, err in ERRORS:
    print(f"  {cid}: {err}")

lines = ["# Run summary — 2026-08-16 (Run 2)", ""]
lines += ["## Headline numbers", "", "| key | value |", "|---|---|"]
for k, v in RESULTS.items():
    lines.append(f"| {k} | {v} |")
lines += ["", "## Errors / skipped cells", ""]
if ERRORS:
    for cid, err in ERRORS:
        lines.append(f"- **{cid}**: {err}")
else:
    lines.append("- none")

(OUT_DIR / "RESULTS_SUMMARY.md").write_text("\n".join(lines))
print(f"\nwrote {OUT_DIR / 'RESULTS_SUMMARY.md'}")